# UIT DSC 2026 LegalIR
Attach the competition-data and cached-artifacts datasets, then set the paths in `configs/kaggle_t4x2.yaml` if needed.

In [ ]:
# Set this to the attached competition-data dataset.
from pathlib import Path
DATASET_DIR = Path('/kaggle/input/REPLACE_WITH_DATASET_SLUG')

In [ ]:
!git clone https://github.com/REPLACE_WITH_YOUR_ORG/UIT-LegalIR.git
%cd UIT-LegalIR
import shutil

# The Kaggle Dataset stores contexts at selected-contexts/selected-contexts/context_*.json.
# Link the repository directory directly to the folder containing the context files.
contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir():
    raise FileNotFoundError(f'Missing context directory: {contexts_source}')
context_count = sum(1 for _ in contexts_source.glob('context_*.json'))
if not context_count:
    raise FileNotFoundError(f'No context_*.json files found in {contexts_source}')

def replace_with_symlink(target, source, *, target_is_directory=False):
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)
    target.symlink_to(source, target_is_directory=target_is_directory)

replace_with_symlink(Path('selected-contexts'), contexts_source, target_is_directory=True)
for item in ('train.json', 'public-official.json'):
    source = DATASET_DIR / item
    if not source.is_file():
        raise FileNotFoundError(f'Missing required dataset file: {source}')
    replace_with_symlink(Path(item), source)

linked_contexts = Path('selected-contexts')
print(f'Using {sum(1 for _ in linked_contexts.glob("context_*.json"))} legal contexts from {linked_contexts.resolve()}')
!pip install -q -e . -r requirements.txt

In [ ]:
import os, subprocess, sys
base = [sys.executable, '-m', 'legalir']
def run(*args, gpu=None):
    env = dict(os.environ)
    if gpu is not None: env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.run(base + list(args), check=True, env=env)

run('prepare', '--config', 'configs/kaggle_t4x2.yaml', '--resume')
run('audit', '--config', 'configs/kaggle_t4x2.yaml')
run('index', '--config', 'configs/kaggle_t4x2.yaml', '--lexical-only', '--resume')
p0 = subprocess.Popen(base + ['index', '--config', 'configs/kaggle_t4x2.yaml', '--model', 'vietlegal_e5', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
p1 = subprocess.Popen(base + ['index', '--config', 'configs/kaggle_t4x2.yaml', '--model', 'vietnamese_embedding', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
p0.wait(); p1.wait()
assert p0.returncode == p1.returncode == 0
run('index', '--config', 'configs/kaggle_t4x2.yaml', '--model', 'nemotron', '--resume', gpu=0)

In [ ]:
run('tune', '--config', 'configs/kaggle_t4x2.yaml', '--resume')
p0 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'train', '--fold', '0', '--engine', 'jina', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
p1 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'train', '--fold', '0', '--engine', 'vietnamese_reranker', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
p0.wait(); p1.wait()
assert p0.returncode == p1.returncode == 0
run('rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'train', '--fold', '0', '--resume')
run('tune', '--config', 'configs/kaggle_t4x2.yaml', '--final', '--fold', '0', '--resume')
run('retrieve', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'public', '--resume')
p0 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'public', '--engine', 'jina', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
p1 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'public', '--engine', 'vietnamese_reranker', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
p0.wait(); p1.wait()
assert p0.returncode == p1.returncode == 0
run('predict', '--config', 'configs/kaggle_t4x2.yaml', '--output', 'submission.json', '--resume')
subprocess.run(['zip', '-j', 'submission.zip', 'submission.json'], check=True)